# 02 - Preprocessing

This notebook builds reusable preprocessing artifacts for NRMS: vocabulary, encoded news titles, GloVe embeddings, and parsed behavior samples.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.append(str(SRC_DIR))

from data_loader import (
    build_vocabulary,
    combine_news,
    create_train_dataloader,
    encode_news_titles,
    load_behaviors,
    load_glove_embeddings,
    load_news,
    parse_history,
    parse_impressions,
    save_processed_artifacts,
    set_seed,
)

set_seed(42)
TRAIN_DIR = PROJECT_ROOT / 'data' / 'MINDsmall_train'
DEV_DIR = PROJECT_ROOT / 'data' / 'MINDsmall_dev'
GLOVE_PATH = PROJECT_ROOT / 'data' / 'glove' / 'glove.6B.300d.txt'
PROCESSED_DIR = PROJECT_ROOT / 'results' / 'processed'

train_news = load_news(TRAIN_DIR / 'news.tsv')
dev_news = load_news(DEV_DIR / 'news.tsv')
train_behaviors = load_behaviors(TRAIN_DIR / 'behaviors.tsv')
train_news.head()

,news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."


## Build Vocabulary

The vocabulary is built from training titles only with `min_frequency=2`, plus `<PAD>` and `<UNK>` special tokens.

In [2]:
vocab = build_vocabulary(train_news, min_frequency=2)
len(vocab), list(vocab.items())[:10]

(20881,
 [('<PAD>', 0),
  ('<UNK>', 1),
  ('to', 2),
  ('in', 3),
  (',', 4),
  ('the', 5),
  (':', 6),
  ("'s", 7),
  ('of', 8),
  ('for', 9)])

The first value is the vocabulary size after filtering out words that appear fewer than two times. The displayed tokens are the most frequent title tokens plus the special `<PAD>` and `<UNK>` entries used for padding and unseen words.

## Encode Titles

Titles are tokenized with NLTK and padded/truncated to 30 tokens, which is the sequence length consumed by the news encoder.

In [3]:
all_news = combine_news(train_news, dev_news)
news_title_map = encode_news_titles(all_news, vocab, max_length=30)
sample_id = next(iter(news_title_map))
sample_id, news_title_map[sample_id], news_title_map[sample_id].shape

('N55528',
 array([    5,  3366,  1230,  1040,     4,   741,  1468,     4,    13,
          741,  7468, 12335,    29,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0]),
 (30,))

The encoded title is a fixed-length vector of token IDs. A shape of `(30,)` confirms that every title will enter the news encoder with the same length, with trailing zeros representing padding.

## Load GloVe Embeddings

Known vocabulary tokens receive their 300-dimensional GloVe vectors. Missing words are randomly initialized and the padding vector is all zeros.

In [4]:
embedding_matrix = load_glove_embeddings(GLOVE_PATH, vocab, embedding_dim=300, seed=42)
embedding_matrix.shape

Loaded GloVe vectors for 19,051/20,881 tokens (91.2%).


(20881, 300)

The embedding matrix shape should be `(vocabulary_size, 300)`. Each row corresponds to one token ID, and each column is one dimension from the 300-dimensional GloVe representation.

## Parse Behaviors

Behaviors are split into user history and impression candidate-label pairs. The training DataLoader samples one positive plus four negatives per positive click.

In [5]:
row = train_behaviors.iloc[0]
parsed = {
    'history': parse_history(row['history'])[:5],
    'impressions': parse_impressions(row['impressions'])[:5],
}
parsed

{'history': ['N55189', 'N42782', 'N34694', 'N45794', 'N18445'],
 'impressions': [('N55689', 1), ('N35729', 0)]}

The parsed history list contains previously clicked article IDs, while the impressions list contains candidate article IDs paired with labels. A label of `1` means clicked and `0` means shown but not clicked.

In [6]:
loader = create_train_dataloader(
    TRAIN_DIR / 'behaviors.tsv',
    news_title_map,
    batch_size=4,
    history_size=50,
    negative_sampling_ratio=4,
    seed=42,
)
history_batch, candidate_batch, label_batch = next(iter(loader))
history_batch.shape, candidate_batch.shape, label_batch

(torch.Size([4, 50, 30]), torch.Size([4, 5, 30]), tensor([4, 1, 1, 0]))

The history batch shape represents users, clicked-history length, and title length. The candidate batch includes one clicked article plus four sampled negatives, and `label_batch` stores the index of the clicked candidate for cross-entropy training.

## Save Processed Data

The encoded title lookup and vocabulary are saved under `results/processed/` for quick reuse in experiments.

In [7]:
save_processed_artifacts(PROCESSED_DIR, vocab, news_title_map)
print(f'Saved preprocessing artifacts to {PROCESSED_DIR}')

Saved preprocessing artifacts to /Users/abhasoli/Documents/Machine Learning/mind-recommender-final/mind-recommender/results/processed
